In [0]:
# Read the raw orders CSV using Auto Loader
# with schema hints for non-string types, schema inference, and schema evolution enabled

df_orders_bronze = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("header", "true") \
    .option("cloudFiles.schemaLocation", "/Volumes/second_data_engineering_project/pipeline_metadata/autoloader_metadata/schemas/bronze/orders") \
    .option("cloudFiles.inferColumnTypes", "true") \
    .option("cloudFiles.schemaHints", "order_purchase_timestamp TIMESTAMP, order_approved_at TIMESTAMP, order_delivered_carrier_date TIMESTAMP, order_delivered_customer_date TIMESTAMP, order_estimated_delivery_date TIMESTAMP") \
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
    .option("rescuedDataColumn", "_rescued_data") \
    .load("/Volumes/second_data_engineering_project/landing/raw_files/olist_orders_dataset")

In [0]:
# Write the streaming Bronze DataFrame as a Delta table
# Using Trigger.AvailableNow for batch-like processing with Auto Loader benefits
# mergeSchema allows the Delta table to accept new columns discovered by Auto Loader
# Checkpoint location enables incremental processing on subsequent runs

df_orders_bronze.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("mergeSchema", "true") \
    .option("checkpointLocation", "/Volumes/second_data_engineering_project/pipeline_metadata/autoloader_metadata/checkpoints/bronze/orders") \
    .trigger(availableNow=True) \
    .toTable("second_data_engineering_project.bronze.orders")

In [0]:
%sql
-- Count rows from the Bronze orders table
SELECT COUNT(*) AS row_count
FROM second_data_engineering_project.bronze.orders;

In [0]:
%sql
SELECT *
FROM second_data_engineering_project.bronze.orders
LIMIT 100;